# Analyze Panoramic Image Sequences with Gemini 3.5 Flash

This notebook demonstrates how to query panoramic Street View imagery directly from BigQuery (`imagery_insights___us.pano_observations_latest`), detect logistical barriers (driveway gates, fences, roadblocks, signage), and **paint 2D bounding boxes and labels directly onto the image** alongside the AI analysis results.

In [ ]:
# @title 1. Installation & Environment Setup
# @markdown Install required libraries and configure project settings.

import os
import io
import json
import google.auth
from google.cloud import bigquery, storage
from google import genai
from google.genai import types
from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import Image as IPImage, display, Markdown
import pandas as pd

# Project and Data Configuration
project_id = "YOUR_PROJECT_ID" #@param {type:"string"}
dataset_id = "imagery_insights___us" #@param {type:"string"}
table_name = "pano_observations_latest" #@param {type:"string"}
limit_count = 10 #@param {type:"integer"}

# Vertex AI Model Configuration
location = "global" #@param {type:"string"}
model_name = "gemini-3.5-flash" #@param {type:"string"}

# Auto-detect GCP project if placeholder is unchanged
if project_id == "YOUR_PROJECT_ID":
    try:
        _, auth_project = google.auth.default()
        if auth_project:
            project_id = auth_project
            print(f"[INFO] Auto-detected Google Cloud Project: {project_id}")
    except Exception:
        pass

print(f"[INFO] Configured project: '{project_id}', table: '{dataset_id}.{table_name}', limit: {limit_count}")

In [ ]:
# @title 2. Data Extraction from BigQuery
# @markdown Fetch panoramic observations directly from the canonical dataset.

client_bq = bigquery.Client(project=project_id)
client_storage = storage.Client(project=project_id)

query = f"""
    SELECT
        observation_id,
        capture_id,
        gcs_uri,
        capture_time
    FROM
        `{project_id}`.`{dataset_id}`.`{table_name}`
    LIMIT {limit_count}
"""

print(f"Executing BigQuery query on `{project_id}`.`{dataset_id}`.`{table_name}`...")
try:
    df_results = client_bq.query(query).to_dataframe()
    if not df_results.empty:
        print(f"[SUCCESS] Retrieved {len(df_results)} panoramic observations.")
        display(df_results.head())
    else:
        print("[WARN] Query returned 0 rows. Please verify table contents.")
except Exception as e:
    print(f"[ERROR] BigQuery query failed: {e}")
    df_results = pd.DataFrame()

In [ ]:
# @title 3. Multimodal Analysis & Visual Bounding Box Painting
# @markdown Detects barriers with 2D bounding boxes and paints annotated overlays directly on the images.

client_genai = genai.Client(vertexai=True, project=project_id, location=location)

analysis_prompt = """
Analyze this street view image for logistics obstructions and hazards.
Detect physical barriers (gates, fences, roadblocks, restricted parking signs, overhead hazards).

Return your response in structured JSON format:
{
  "detected_objects": [
    {
      "label": "Gate/Fence/Roadblock/Sign/Obstruction",
      "box_2d": [ymin, xmin, ymax, xmax],
      "description": "Short description of the object"
    }
  ],
  "summary": "1-2 sentence overall summary of logistics obstacles or 'None'"
}
Note: box_2d coordinates must be normalized integers from 0 to 1000.
"""

def annotate_image(image_bytes, detections):
    """Paints colored bounding boxes and label tags directly on the image."""
    image = PILImage.open(io.BytesIO(image_bytes)).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    
    for det in detections:
        box = det.get("box_2d")
        label = det.get("label", "Obstruction")
        if box and len(box) == 4:
            ymin, xmin, ymax, xmax = box
            left = int((xmin / 1000.0) * width)
            top = int((ymin / 1000.0) * height)
            right = int((xmax / 1000.0) * width)
            bottom = int((ymax / 1000.0) * height)
            
            draw.rectangle([left, top, right, bottom], outline="#FF0055", width=5)
            banner_height = 24
            banner_width = len(label) * 11 + 16
            draw.rectangle([left, max(0, top - banner_height), left + banner_width, top], fill="#FF0055")
            draw.text((left + 6, max(0, top - banner_height + 4)), label, fill="white")
            
    buf = io.BytesIO()
    image.save(buf, format="JPEG", quality=95)
    return buf.getvalue()

results_with_imagery = []

if 'df_results' in locals() and not df_results.empty:
    for idx, row in df_results.iterrows():
        gcs_uri = row['gcs_uri']
        obs_id = row['observation_id']
        print(f"Processing Observation: {obs_id}...")
        
        # 1. Download image bytes
        try:
            path_parts = gcs_uri[5:].split('/', 1)
            bucket = client_storage.bucket(path_parts[0])
            blob = bucket.blob(path_parts[1])
            raw_image_bytes = blob.download_as_bytes()
        except Exception as e:
            print(f"[WARN] Could not download {gcs_uri}: {e}")
            continue
            
        # 2. Call Gemini 3.5 Flash
        try:
            response = client_genai.models.generate_content(
                model=model_name,
                contents=[
                    types.Part.from_bytes(data=raw_image_bytes, mime_type="image/jpeg"),
                    analysis_prompt
                ],
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.1
                )
            )
            parsed = json.loads(response.text)
            detections = parsed.get("detected_objects", [])
            annotated_bytes = annotate_image(raw_image_bytes, detections)
            
            results_with_imagery.append({
                "observation_id": obs_id,
                "gcs_uri": gcs_uri,
                "annotated_bytes": annotated_bytes,
                "detections": detections,
                "summary": parsed.get("summary", "Completed")
            })
        except Exception as e:
            print(f"[ERROR] Failed to analyze {obs_id}: {e}")
            
    print("\n[SUCCESS] Completed analysis and bounding box rendering.")
else:
    print("[WARN] No data available for analysis.")

In [ ]:
# @title 4. Visual Insights & Annotated Imagery Report
# @markdown Displays the annotated Street View images alongside the AI detection results.

if results_with_imagery:
    display(Markdown("## 🚚 Fleet Logistics & Barrier Detection Report"))
    
    for entry in results_with_imagery:
        display(Markdown(f"### Observation: `{entry['observation_id']}`"))
        display(Markdown(f"* **GCS URI**: `{entry['gcs_uri']}`"))
        display(Markdown(f"* **AI Summary**: {entry['summary']}"))
        
        # Display annotated image with bounding boxes painted directly on it
        display(IPImage(data=entry['annotated_bytes'], width=650))
        
        if entry['detections']:
            display(Markdown("**Detected Bounding Boxes:**"))
            for det in entry['detections']:
                display(Markdown(f"- 🏷️ **{det.get('label', 'Item')}**: {det.get('description', '')} (Box: `{det.get('box_2d')}`)"))
        display(Markdown("---"))
else:
    print("No results to display.")